# IMPORT THƯ VIỆN VÀ DEF FUNCTION

In [2]:
import gc
import re
from pathlib import Path
import polars as pl

# -----------------------------------------------------------------------------
# HÀM XỬ LÝ CHUỖI NGÀY THÁNG (ĐÃ TỐI ƯU & BẢO VỆ)
# -----------------------------------------------------------------------------
def parse_multi_date(col_name: str) -> pl.Expr:
    col_expr = pl.col(col_name)
    
    # 1. Chuẩn hóa chuỗi (chỉ áp dụng nếu là chuỗi)
    clean_str = (
        col_expr
        .cast(pl.String)
        .str.strip_chars()
        .str.replace_all("/", "-")
    )

    # 2. Parse đa định dạng
    parsed_datetime = pl.coalesce([
        # Nếu cột vốn đã là Datetime/Date thì giữ nguyên
        col_expr.cast(pl.Datetime, strict=False),
        
        # Parse chuỗi dạng Năm - Ngày - Tháng
        clean_str.str.to_datetime("%Y-%d-%m %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%Y-%d-%m %H:%M", strict=False),
        clean_str.str.to_datetime("%Y-%d-%m", strict=False),

        # Parse chuỗi dạng Năm - Tháng - Ngày
        clean_str.str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%Y-%m-%d %H:%M", strict=False),
        clean_str.str.to_datetime("%Y-%m-%d", strict=False),

        # Parse chuỗi dạng Ngày - Tháng - Năm (Việt Nam)
        clean_str.str.to_datetime("%d-%m-%Y %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%d-%m-%Y %H:%M", strict=False),
        clean_str.str.to_datetime("%d-%m-%Y", strict=False),
    ])

    return parsed_datetime

def ultimate_to_date(col_name, df_context):
    if col_name not in df_context.columns:
        return pl.lit(None, dtype=pl.Date)

    col_str = pl.col(col_name).cast(pl.String)

    # Đọc chuỗi (dù là ISO, YYYY-MM-DD HH:MM:SS hay DD/MM/YYYY) và ép thẳng về Date
    parsed_date = pl.coalesce([
        col_str.str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
        col_str.str.slice(0, 10).str.to_date("%d/%m/%Y", strict=False),
        col_str.str.slice(0, 10).str.to_date("%d-%m-%Y", strict=False),
    ])

    excel_date = (
        pl.col(col_name)
        .cast(pl.Float64, strict=False)
        .cast(pl.Duration("ms"))
        + pl.date(1899, 12, 30)
    ).dt.date()

    return pl.coalesce([parsed_date, excel_date])

# -----------------------------------------------------------------------------
# HÀM XỬ LÝ DATETIME AN TOÀN (CHỐNG TRÀN SỐ & NESTED OBJECT TYPES)
# -----------------------------------------------------------------------------
def ultimate_to_datetime(col_name, df_context):
    if col_name not in df_context.collect_schema().names():
        return pl.lit(None, dtype=pl.Datetime)

    col_expr = pl.col(col_name)
    col_str = col_expr.cast(pl.String, strict=False)
    col_str_19 = col_str.str.slice(0, 19)

    # 1. Parse các định dạng chuỗi ngày tháng phổ biến
    parsed_dt = pl.coalesce([
        col_str_19.str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
        col_str.str.to_datetime("%Y-%m-%d", strict=False),
        col_str_19.str.to_datetime("%d/%m/%Y %H:%M:%S", strict=False),
        col_str.str.to_datetime("%d/%m/%Y", strict=False),
        col_str_19.str.to_datetime("%d-%m-%Y %H:%M:%S", strict=False),
        col_str.str.to_datetime("%d-%m-%Y", strict=False),
    ])

    # 2. Xử lý Serial Date của Excel an toàn tuyệt đối (có định nghĩa time_unit="ms")
    excel_dt = (
        (col_expr.cast(pl.Float64, strict=False) * 86_400_000)
        .cast(pl.Duration("ms"), strict=False)
        + pl.datetime(1899, 12, 30, time_unit="ms")
    )

    # Trả về kết quả ưu tiên chuỗi, nếu lỗi/null thì lấy kiểu số Excel, nếu vẫn lỗi trả về null
    return pl.coalesce([parsed_dt, excel_dt])

# Cleanned SPE data phát

In [ ]:
# -----------------------------------------------------------------------------
# LUỒNG XỬ LÝ CHÍNH (CHỐNG TRÀN RAM 100%)
# -----------------------------------------------------------------------------
path_spe_path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\Phat")
temp_parquet_dir = path_spe_path / "temp_parquet_cache"
temp_parquet_dir.mkdir(parents=True, exist_ok=True)

print("🚀 Đang nạp và chuyển đổi file Excel sang Parquet cục bộ...")

excel_files_phat = list(path_spe_path.glob("chitietketquaphat_TikTok_*.xlsx"))

if not excel_files_phat:
    print("❌ Không tìm thấy file Excel nào!")
else:
    for i, file in enumerate(excel_files_phat):
        print(f"📄 Đang đọc file [{i+1}/{len(excel_files_phat)}]: {file.name}")
        try:
            # Dùng calamine để đọc Excel siêu nhẹ
            df = pl.read_excel(file, drop_empty_rows=True, engine="calamine")
            
            # Ép kiểu an toàn (trừ cột tg_ptc ra để dữ liệu ngày tháng không bị hỏng trước khi parse)
            df = df.with_columns([
                pl.col(c).cast(pl.String) for c in df.columns if c != "tg_ptc"
            ])
            
            # Ghi ngay ra file parquet tạm trên ổ đĩa để giải phóng RAM lập tức
            temp_file_path = temp_parquet_dir / f"temp_{i}.parquet"
            df.write_parquet(temp_file_path)
            
            # Xóa biến và dọn RAM
            del df
            gc.collect()
            
        except Exception as e:
            print(f"⚠️ Lỗi ở file {file.name}: {e}")

    print("⚡ Đang quét dữ liệu qua ổ đĩa (Streaming Safe)...")
    
    # Quét dữ liệu bằng Lazy API qua ổ đĩa
    dfs_spe_phat_final = pl.scan_parquet(str(temp_parquet_dir / "*.parquet"))

    # Tự động nhận diện cột ngày tháng từ schema mẫu
    sample_cols = pl.read_parquet(str(temp_parquet_dir / "temp_0.parquet"), n_rows=1).columns
    date_cols = [
        c for c in sample_cols 
        if any(k in c.lower() for k in ["ngay", "time", "tg", "thoigian"]) 
        and c.lower() != "tg_quydinh"
    ]

    # Áp dụng hàm ultimate_to_datetime cho các cột ngày tháng
    if date_cols:
        for c in date_cols:
            if c in dfs_spe_phat_final.collect_schema().names():
                dfs_spe_phat_final = dfs_spe_phat_final.with_columns(
                    ultimate_to_datetime(c, dfs_spe_phat_final).alias(c)
                )

    # Gom kết quả cuối cùng với streaming mode
    dfs_spe_phat_final_collected = dfs_spe_phat_final.collect(streaming=True)

    # Dọn dẹp sạch sẽ các file tạm
    for p in temp_parquet_dir.glob("*.parquet"):
        p.unlink()
    temp_parquet_dir.rmdir()

    print("=" * 70)
    print(f"✅ GỘP THÀNH CÔNG! Tổng số dòng: {dfs_spe_phat_final_collected.height:,}")
    print("=" * 70)

In [ ]:
df_spe_phat_processed = dfs_spe_phat_final_collected.with_columns([
    # Ép kiểu trọng lượng về số và phân nhóm
    pl.when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 5000)
    .then(pl.lit("< 5kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 10000)
    .then(pl.lit("5kg - < 10kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 20000)
    .then(pl.lit("10kg - < 20kg"))
    .otherwise(pl.lit(">= 20kg"))
    .alias("nhom_trong_luong")
])

# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file = "SPE_phat_new_today.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh = Path(path_spe_path) / ten_file
duong_dan_hoan_chinh.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_spe_phat_processed.write_parquet(duong_dan_hoan_chinh)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh}")

In [ ]:
df_gop = pl.scan_parquet(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\phat\SPE_phat*.parquet").collect()

SPE_file = "SPE_phat_database.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
spe_phat_duong_dan = Path(path_spe_path) / SPE_file
spe_phat_duong_dan.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_gop.write_parquet(spe_phat_duong_dan)

print(f"Đã lưu file Parquet thành công tại: {spe_phat_duong_dan}")

# Cleanned TTS data phát

In [3]:
# -----------------------------------------------------------------------------
# LUỒNG XỬ LÝ CHÍNH (CHỐNG TRÀN RAM 100%)
# -----------------------------------------------------------------------------
path_tts_path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Phat")
temp_parquet_dir = path_tts_path / "temp_parquet_cache"
temp_parquet_dir.mkdir(parents=True, exist_ok=True)

print("🚀 Đang nạp và chuyển đổi file Excel sang Parquet cục bộ...")

excel_files_phat = list(path_tts_path.glob("chitietketquaphat_TikTok_*.xlsx"))

if not excel_files_phat:
    print("❌ Không tìm thấy file Excel nào!")
else:
    for i, file in enumerate(excel_files_phat):
        print(f"📄 Đang đọc file [{i+1}/{len(excel_files_phat)}]: {file.name}")
        try:
            # Dùng calamine để đọc Excel siêu nhẹ
            df = pl.read_excel(file, drop_empty_rows=True, engine="calamine")
            
            # Ép kiểu an toàn (trừ cột tg_ptc ra để dữ liệu ngày tháng không bị hỏng trước khi parse)
            df = df.with_columns([
                pl.col(c).cast(pl.String) for c in df.columns if c != "tg_ptc"
            ])
            
            # Ghi ngay ra file parquet tạm trên ổ đĩa để giải phóng RAM lập tức
            temp_file_path = temp_parquet_dir / f"temp_{i}.parquet"
            df.write_parquet(temp_file_path)
            
            # Xóa biến và dọn RAM
            del df
            gc.collect()
            
        except Exception as e:
            print(f"⚠️ Lỗi ở file {file.name}: {e}")

    print("⚡ Đang quét dữ liệu qua ổ đĩa (Streaming Safe)...")
    
    # Quét dữ liệu bằng Lazy API qua ổ đĩa
    dfs_tts_phat_final = pl.scan_parquet(str(temp_parquet_dir / "*.parquet"))

    # Tự động nhận diện cột ngày tháng từ schema mẫu
    sample_cols = pl.read_parquet(str(temp_parquet_dir / "temp_0.parquet"), n_rows=1).columns
    date_cols = [
        c for c in sample_cols 
        if any(k in c.lower() for k in ["ngay", "time", "tg", "thoigian"]) 
        and c.lower() != "tg_quydinh"
    ]

    # Áp dụng hàm ultimate_to_datetime cho các cột ngày tháng
    if date_cols:
        for c in date_cols:
            if c in dfs_tts_phat_final.collect_schema().names():
                dfs_tts_phat_final = dfs_tts_phat_final.with_columns(
                    ultimate_to_datetime(c, dfs_tts_phat_final).alias(c)
                )

    # Gom kết quả cuối cùng với streaming mode
    dfs_tts_phat_final_collected = dfs_tts_phat_final.collect(streaming=True)

    # Dọn dẹp sạch sẽ các file tạm
    for p in temp_parquet_dir.glob("*.parquet"):
        p.unlink()
    temp_parquet_dir.rmdir()

    print("=" * 70)
    print(f"✅ GỘP THÀNH CÔNG! Tổng số dòng: {dfs_tts_phat_final_collected.height:,}")
    print("=" * 70)

🚀 Đang nạp và chuyển đổi file Excel sang Parquet cục bộ...
📄 Đang đọc file [1/1]: chitietketquaphat_TikTok_18092026.xlsx


Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 32, falling back to string
Could not determine dtype for column 35, falling back to string
Could not determine dtype for column 36, falling back to string
Could not determine dtype for column 37, falling back to string
Could not determine dtype for column 38, falling back to string


⚡ Đang quét dữ liệu qua ổ đĩa (Streaming Safe)...


C:\Users\lamnv5_vtp\AppData\Local\Temp\ipykernel_1228\2516640742.py:59: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  dfs_tts_phat_final_collected = dfs_tts_phat_final.collect(streaming=True)


✅ GỘP THÀNH CÔNG! Tổng số dòng: 448,235


In [4]:
df_tts_phat_processed = dfs_tts_phat_final_collected.with_columns([
    # Ép kiểu trọng lượng về số và phân nhóm
    pl.when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 5000)
    .then(pl.lit("< 5kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 10000)
    .then(pl.lit("5kg - < 10kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 20000)
    .then(pl.lit("10kg - < 20kg"))
    .otherwise(pl.lit(">= 20kg"))
    .alias("nhom_trong_luong")
])

# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file = "TTS_phat_new_today.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh = Path(path_tts_path) / ten_file
duong_dan_hoan_chinh.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_tts_phat_processed.write_parquet(duong_dan_hoan_chinh)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh}")

Đã lưu file Parquet thành công tại: C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Phat\TTS_phat_new_today.parquet


In [6]:
import os

# --- 1. Khai báo đường dẫn và tên file ---
path_tts_path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\phat")
TTS_file = "TTS_phat_database.parquet"

tts_phat_duong_dan = path_tts_path / TTS_file
tts_phat_duong_dan.parent.mkdir(parents=True, exist_ok=True)

# File tạm để ghi dữ liệu (tránh lỗi khóa file os error 1224)
tts_phat_temp_path = path_tts_path / "TTS_phat_database_TEMP.parquet"

# --- 2. Lấy danh sách file NGUỒN ---
all_parquet_files = list(path_tts_path.glob("TTS_phat*.parquet"))
source_files = [
    str(f) for f in all_parquet_files 
    if f.name not in [TTS_file, tts_phat_temp_path.name]
]

# --- 3. Đọc, gộp & LỌC TRÙNG (Sửa tùy chọn datetime_cast) ---
if source_files:
    if tts_phat_duong_dan.exists():
        source_files.append(str(tts_phat_duong_dan))

    df_gop_2 = (
        pl.scan_parquet(
            source_files,
            # CHÚ Ý THAY ĐỔI Ở ĐÂY: Dùng microsecond-downcast để đưa μs về ms
            cast_options=pl.ScanCastOptions(datetime_cast="microsecond-downcast")
        )
        .unique() # Xóa dòng trùng lặp
        .collect(streaming=True) # Tiết kiệm RAM
    )

    # --- 4. Ghi ra file tạm rồi đổi tên ---
    df_gop_2.write_parquet(tts_phat_temp_path)

    # Đổi tên file tạm ghi đè file chính
    os.replace(tts_phat_temp_path, tts_phat_duong_dan)

    print(f"Đã lưu file Parquet thành công tại: {tts_phat_duong_dan}")
else:
    print("Không tìm thấy file nguồn mới nào để gộp.")

C:\Users\lamnv5_vtp\AppData\Local\Temp\ipykernel_1228\439651076.py:32: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True) # Tiết kiệm RAM


Đã lưu file Parquet thành công tại: C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\phat\TTS_phat_database.parquet
